# Konu 6 — Üretmek ne demek?

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Bu defter **anahtar istemez**; internete de çıkmaz. Bütün iş senin bilgisayarında.

Bugünün sorusu: bir model hiç görmediği bir şeyi nasıl "üretir"?
Cevabı 136 farklı fontta yazılmış **A** harfiyle bulacağız.

## Isınma — Bir resim, bir sayı tablosudur

Konu 3'te renk üç sayıydı. Siyah-beyaz bir resim de sayılardan oluşur: her kare (piksel) bir sayı.
**0** boş, **1** mürekkep. Aşağıdaki tablo 5 satır, 5 sütun.

`plt.imshow` bir sayı tablosunu resim olarak çizer. `cmap="gray_r"`: 0 beyaz, 1 siyah.

In [ ]:
import matplotlib.pyplot as plt

arti = [
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
]
plt.imshow(arti, cmap="gray_r")
plt.show()

Tablodaki sayıları değiştirip yeniden çalıştır:

- Artıyı bir **T** harfine çevir: hangi satırları değiştirmen gerekiyor?
- Bir kareye `0.5` yaz: ne oluyor?

Bilgisayar için bir harf, bir fotoğraf, bir afiş: hepsi böyle sayı tabloları. Bugünkü harfler 28 satır, 28 sütun:
her biri **784** sayı.

## Adım 1 — 136 fontta A harfi

Veri dosyası `veri/harfler.npz` bu konunun klasöründe. İçinde 136 fontta yazılmış A, a, g ve R harfleri var.
`np.load` dosyayı açar; `veri["A"]` A harflerini verir (sözlükten değer alır gibi). NumPy'ı Konu 4'ün Isınma 2'sinden hatırla.

Dosyada her piksel Konu 3'teki RGB gibi 0 ile 255 arasında. `/ 255` hepsini 0 ile 1 arasına çeker: ısınmadaki gibi.
`list(...)` font adlarını ve türlerini bildiğimiz Python listesine çevirir; ikisini Adım 5 ve 8'de kullanacağız.

In [ ]:
import numpy as np

veri = np.load("veri/harfler.npz")
resimler = veri["A"] / 255
fontlar = list(veri["fontlar"])
turler = list(veri["turler"])

print(resimler.shape)
print(fontlar[0], turler[0])

`(136, 784)`: 136 satır, her satırda 784 sayı. Her satır bir fontun A'sı; Konu 3'teki tablonun aynısı, yalnızca sütunlar çok daha fazla.

784 sayı yan yana duruyor; resim olarak çizmek için onları 28 satıra bölmemiz gerekiyor.
`reshape(28, 28)` tam bunu yapar: "28 satır, 28 sütun olarak diz."

In [ ]:
ilk = resimler[0]
plt.imshow(ilk.reshape(28, 28), cmap="gray_r")
plt.show()

Bir de ilk sekizine bakalım. `plt.subplot(1, 8, sira)`: "1 satır, 8 sütunluk bir ızgarada `sira` numaralı kutuya çiz".
Kutular 1'den başlar. Sayaç `sira` her turda bir artıyor; Konu 1'deki sayacın aynısı.

In [ ]:
sira = 1
for resim in resimler[:8]:
    plt.subplot(1, 8, sira)
    plt.imshow(resim.reshape(28, 28), cmap="gray_r")
    plt.axis("off")
    sira = sira + 1
plt.show()

Aynı harf, sekiz farklı karakter: kalın, eğik, ince, serifli. Bir tipograf bu farkları adlandırır. Bilgisayar ise yalnızca 784 sayı görüyor.

## Adım 2 — Sıkıştır: 784 sayıdan 2 sayıya

Konu 4'te PCA ile yüzlerce sayıyı 2 sayıya indirip harita çizmiştik. Aynısını harflere yapıyoruz.

- `fit`: PCA 136 A'nın hepsine bakıp en çok şey anlatan iki yönü bulur (Konu 4'teki gibi).
- `transform`: her A'yı **2 sayıya** sıkıştırır. Bu iki sayıya harfin **kodu** diyelim.

Buradaki "kod", yazdığımız Python kodu değil: harfin sıkıştırılmış hâli, kısa bir özeti.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(resimler)
kodlar = pca.transform(resimler)

print(kodlar.shape)
print(kodlar[0])

`(136, 2)`: artık her A yalnızca **iki sayı**. 784 sayıdan 2 sayıya: neredeyse 400 kat sıkıştırdık.

## Adım 3 — Geri kur: 2 sayıdan 784 sayıya

Konu 4'te burada durmuştuk. Bugün bir adım daha atıyoruz: **geri dönüyoruz.**
`inverse_transform`, `transform`'un tersi: 2 sayılık koddan 784 sayılık bir resim kurar.

**Tahmin et, sohbete yaz:** yalnızca 2 sayıdan geri kurulan A nasıl görünecek? Okunur mu, bir leke mi?

In [ ]:
geri = pca.inverse_transform(kodlar)

plt.subplot(1, 2, 1)
plt.imshow(resimler[0].reshape(28, 28), cmap="gray_r")
plt.title("asıl")
plt.subplot(1, 2, 2)
plt.imshow(geri[0].reshape(28, 28), cmap="gray_r")
plt.title("2 sayıdan")
plt.show()

Bulanık ama bir A. Yalnızca 2 sayı saklayıp geri kalan 782'yi **PCA'nın öğrendiklerinden** doldurduk.

Bu ikiliye **sıkıştır ve geri kur** diyoruz: bir parça (kodlayıcı) resmi küçük bir koda sıkıştırır,
öbür parça (çözücü) koddan resmi geri kurar. Görsel üreten büyük modeller de bu ikiliyi kullanır.

## Adım 4 — Kaç sayı yeter?

2 sayı az geldi. Aynı A'yı 1, 2, 5, 10, 40 ve 100 sayıdan geri kuralım.
Döngü her turda yeni bir PCA kuruyor; tek fark `n_components`.

**Tahmin et:** kaç sayıda A asıl hâline "yeterince" benzer?

In [ ]:
sira = 1
for boyut in [1, 2, 5, 10, 40, 100]:
    pca_dene = PCA(n_components=boyut)
    pca_dene.fit(resimler)
    kod_dene = pca_dene.transform(resimler)
    geri_dene = pca_dene.inverse_transform(kod_dene)
    plt.subplot(1, 6, sira)
    plt.imshow(geri_dene[0].reshape(28, 28), cmap="gray_r")
    plt.title(boyut)
    plt.axis("off")
    sira = sira + 1
plt.show()

Sayı arttıkça ayrıntı geliyor; ama 40'tan sonra gözle fark kalmıyor. Bir JPEG dosyasının "kalite" ayarı da buna benzer:
daha az sayı, daha küçük dosya, daha bulanık resim.

Peki neden 2'de kalıyoruz? Çünkü 2 sayı bir **harita** demek: kâğıda çizebiliriz.

## Adım 5 — Harita: her A bir nokta

Her A'nın 2 sayılık kodunu bir nokta olarak çiziyoruz. Bu haritaya **gizli uzay** (latent space) denir: "gizli", çünkü eksenlerinin adı yok.
Renk, fontun türü:

| Renk | Tür |
|---|---|
| kırmızı | serif: harf uçlarında küçük tırnaklar var |
| mavi | sans: tırnaksız |
| yeşil | mono: her harf aynı genişlikte, daktilo gibi |
| mor | el yazısı (yalnızca 2 font: haritada 2 nokta) |

`renkler[tur]` sözlükten o türün rengini alır: `renkler["serif"]` → `"red"`.
Bu kez sayaç `sira` 0'dan başlıyor: `turler[sira]` ile listeden okuyoruz ve listede sıra 0'dan başlar
(`subplot`'un kutuları ise 1'den başlıyordu).

**Tahmin et, sohbete yaz:** harita serifleri bir köşeye, sans'ları başka bir köşeye toplayacak mı?

In [ ]:
renkler = {"serif": "red", "sans": "blue", "mono": "green", "el yazısı": "purple"}

sira = 0
for nokta in kodlar:
    tur = turler[sira]
    plt.scatter(nokta[0], nokta[1], color=renkler[tur])
    sira = sira + 1

plt.xlabel("kodun 1. sayısı")
plt.ylabel("kodun 2. sayısı")
plt.show()

Kırmızılar ile maviler karışık: harita serif ile sans'ı **ayırmadı**. Ama noktalar rastgele de değil;
solda ve sağda iki büyük öbek var. Harita A'ları başka bir şeye göre dizmiş. Neye göre?
Adım 7'de haritanın kendisini "çizerek" bakacağız.

## Adım 6 — Haritada bir nokta seç, bir A üret

Haritada istediğimiz bir noktayı seçip `inverse_transform` ile geri kurabiliriz.
`[[0, 0]]`: içinde tek bir nokta olan bir liste (`transform` gibi `inverse_transform` da liste ister).
Cevap da liste içinde gelir: `yeni[0]`.

In [ ]:
nokta = [[0, 0]]
yeni = pca.inverse_transform(nokta)

plt.imshow(yeni[0].reshape(28, 28), cmap="gray_r")
plt.show()

Bu A, 136 fontun **hiçbirinde yok.** Haritanın ortasındaki boş bir yerden geldi. İşte **üretmek** bu:
öğrenilmiş bir haritada bir nokta seçip oradan geri kurmak.

Noktayı değiştirip yeniden çalıştır: `[[4, 4]]`, `[[-4, -4]]`, `[[4, -4]]`.
Slaytta haritanın kenarlarını konuştuk. `[[20, 20]]` ile kendin gör: harf nasıl bozuluyor, zemine ne oluyor?

## Adım 7 — Haritanın eksenleri: iki yönde yürü

Haritanın ortasından (0, 0) başlayıp önce **soldan sağa** yürüyoruz: ikinci sayı 0'da sabit, birinci sayı −4'ten 4'e.
Her durakta bir A geri kuruyoruz.

**Tahmin et, sohbete yaz:** soldan sağa giderken A'da ne değişecek?

In [ ]:
sira = 1
for x in [-4, -2, 0, 2, 4]:
    yeni = pca.inverse_transform([[x, 0]])
    plt.subplot(1, 5, sira)
    plt.imshow(yeni[0].reshape(28, 28), cmap="gray_r")
    plt.axis("off")
    sira = sira + 1
plt.show()

Soldan sağa A **eğiliyor** (italik). Şimdi **yukarıdan aşağı**: birinci sayı 0'da sabit, ikinci sayı 4'ten −4'e.

In [ ]:
sira = 1
for y in [4, 2, 0, -2, -4]:
    yeni = pca.inverse_transform([[0, y]])
    plt.subplot(1, 5, sira)
    plt.imshow(yeni[0].reshape(28, 28), cmap="gray_r")
    plt.axis("off")
    sira = sira + 1
plt.show()

Yukarıdan aşağı A **inceliyor**: yukarıda kalın, aşağıda ince. En alttaki A'nın zemini griye döndü: PCA orada bazı piksellere
"eksi mürekkep" veriyor. Düz çizgilerle çalışan basit haritamız ince A'ları o kadar iyi kuramıyor.

Kimse PCA'ya "eğiklik" ya da "kalınlık" demedi. 136 A'ya bakıp en çok bu iki bakımdan farklılaştıklarını **kendisi buldu**.
Adım 5'teki iki öbek: solda dik A'lar, sağda eğik A'lar.

Serif ile sans neden ayrılmadı? Hem serifli hem serifsiz fontların dik ve eğik hâlleri var. PCA bizim kategorilerimize değil,
**pikseller** arasındaki en büyük farka bakıyor; o fark da eğiklik çıktı.

## Adım 8 — İki font arasında yürümek

Dik, ince, serifli bir A ile kalın, eğik, serifsiz bir A seçiyoruz. `fontlar.index(...)` bir fontun listedeki sırasını verir.
`ok = son - bas`: baştan sona giden ok (Konu 4'ün Isınma 2'sindeki çıkarma).

`bas + oran * ok`: okun `oran` kadarını yürü. `oran * ok` oku kısaltır: 0.5 okun yarısı demek.
`oran` 0 iken baştasın, 1 iken sondasın. `ara` zaten iki sayı; `[ara]` onu `[[x, y]]` gibi tek noktalık bir listeye koyar.

In [ ]:
dik = fontlar.index("Liberation Serif Regular")
egik = fontlar.index("Lato Black Italic")
bas = kodlar[dik]
son = kodlar[egik]
ok = son - bas

sira = 1
for oran in [0, 0.25, 0.5, 0.75, 1]:
    ara = bas + oran * ok
    yeni = pca.inverse_transform([ara])
    plt.subplot(1, 5, sira)
    plt.imshow(yeni[0].reshape(28, 28), cmap="gray_r")
    plt.title(oran)
    plt.axis("off")
    sira = sira + 1
plt.show()

Ortadaki üç A, iki fontun arasındaki **ara formlar**. Hiçbiri gerçek bir font değil.
Buna **ara değerleme** diyoruz: haritada iki nokta arasında yürümek.

Bizim haritamız basit (PCA), o yüzden ara formlar iki harfin üst üste binmiş hâli gibi görünüyor.
Görsel üreten büyük modellerin haritası çok daha zengin: orada iki yüz, iki ürün, iki üslup arasında yürüyünce
gerçekten yeni, inandırıcı biçimler çıkar.

## Bonus — Başka bir harf

Adım 1'deki ilk hücrede `veri["A"]` yazan yeri `veri["g"]` yap, hücreleri **Adım 1'den başlayarak** sırayla yeniden çalıştır.
(Adım 8'deki iki font adı da dosyada var; değiştirmen gerekmez.) Adım 2'yi atlama: atlarsan hata çıkmaz ama PCA hâlâ
A'ları öğrenmiş olur ve harita A'nın haritası kalır.

**Tahmin et:** küçük g'de Adım 7'nin soldan sağa yürüyüşü neyi değiştirecek?
İpucu: bazı fontlarda g iki katlıdır (üstte bir halka, altta bir halka), bazılarında tek katlı (alttaki kanca).

Sonra `veri["a"]` ve `veri["R"]` dene.